# Analyse du cours de NVIDIA (NVDA) — Machine Learning
**Adaptation du notebook de référence à `NVIDIA_STOCK.csv`**

Ce notebook reprend les familles et l'enchaînement des 29 algorithmes du notebook fourni, en les appliquant aux données historiques NVIDIA. Les expériences prédictives utilisent une séparation chronologique (pas de mélange aléatoire) afin de respecter l'ordre temporel.

**Important :** les résultats sont pédagogiques et historiques. Ils ne constituent ni un conseil d'investissement ni une garantie de performance future.

In [3]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, classification_report,
                             confusion_matrix, silhouette_score)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
    HistGradientBoostingRegressor, HistGradientBoostingClassifier,
    GradientBoostingClassifier, AdaBoostClassifier, IsolationForest)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from sklearn.inspection import permutation_importance
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Chargement et nettoyage des données

In [4]:
# Le fichier NVIDIA_STOCK.csv doit se trouver dans le même dossier que ce notebook.
df_raw = pd.read_csv("NVIDIA_STOCK.csv")
# Retirer les deux lignes d'en-tête supplémentaires présentes dans l'export.
df = df_raw.iloc[2:].copy()
df = df.rename(columns={"Price": "Date"})
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
numeric_cols = ["Adj Close", "Close", "High", "Low", "Open", "Volume"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Date"] + numeric_cols).sort_values("Date")
df = df.drop_duplicates(subset="Date").set_index("Date")
df = df[numeric_cols]
print("Dimensions :", df.shape)
print("Période :", df.index.min().date(), "->", df.index.max().date())
display(df.head())
display(df.describe().T)

FileNotFoundError: [Errno 2] No such file or directory: 'NVIDIA_STOCK.csv'

## 2. Exploration descriptive

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
df["Adj Close"].plot(ax=axes[0], title="NVIDIA — cours ajusté")
df["Volume"].plot(ax=axes[1], title="Volume quotidien", color="tab:orange")
plt.tight_layout()
plt.show()

In [ ]:
# Vérifications qualité
print("Valeurs manquantes :", df.isna().sum())
print("Lignes dupliquées :", df.index.duplicated().sum())
print("Rendement journalier (%) :")
display((df["Adj Close"].pct_change()*100).describe())

## 3. Construction des variables et cible

In [5]:
data = df.copy()
data["return_1d"] = data["Adj Close"].pct_change()
data["return_5d"] = data["Adj Close"].pct_change(5)
data["volatility_5d"] = data["return_1d"].rolling(5).std()
data["volatility_20d"] = data["return_1d"].rolling(20).std()
data["ma_ratio_5"] = data["Adj Close"] / data["Adj Close"].rolling(5).mean() - 1
data["ma_ratio_20"] = data["Adj Close"] / data["Adj Close"].rolling(20).mean() - 1
data["volume_change"] = data["Volume"].pct_change()
data["range_pct"] = (data["High"] - data["Low"]) / data["Close"]
# La cible est le rendement ajusté du prochain jour de cotation.
data["target_return"] = data["Adj Close"].pct_change().shift(-1)
data["target_up"] = (data["target_return"] > 0).astype(int)
features = ["return_1d","return_5d","volatility_5d","volatility_20d",
            "ma_ratio_5","ma_ratio_20","volume_change","range_pct"]
model_data = data[features + ["target_return","target_up"]].replace([np.inf,-np.inf], np.nan).dropna()
X = model_data[features]
y_reg = model_data["target_return"]
y_cls = model_data["target_up"]
cut = int(len(model_data)*0.8)
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
yr_train, yr_test = y_reg.iloc[:cut], y_reg.iloc[cut:]
yc_train, yc_test = y_cls.iloc[:cut], y_cls.iloc[cut:]
print("Observations utilisables :", len(model_data))
print("Entraînement :", len(X_train), "| Test chronologique :", len(X_test))
print("Variables :", features)

NameError: name 'df' is not defined

## 4. Évaluation de référence
Les modèles sont évalués sur la dernière tranche chronologique. Les transformations de mise à l'échelle sont intégrées aux pipelines et ajustées uniquement sur l'entraînement.

In [ ]:
def eval_reg(model, name):
    model.fit(X_train, yr_train)
    pred = model.predict(X_test)
    rmse = mean_squared_error(yr_test, pred)**0.5
    print(f"{name} | MAE={mean_absolute_error(yr_test,pred):.6f} | RMSE={rmse:.6f} | R²={r2_score(yr_test,pred):.4f}")
    return model, pred

def eval_cls(model, name):
    model.fit(X_train, yc_train)
    pred = model.predict(X_test)
    print(f"{name} | Accuracy={accuracy_score(yc_test,pred):.4f} | F1={f1_score(yc_test,pred,zero_division=0):.4f}")
    return model, pred

## 1. Régression linéaire

Régression supervisée. Elle fournit une référence linéaire pour le rendement du prochain jour.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m1, p1 = eval_reg(LinearRegression(), 'Régression linéaire')

## 2. Ridge

Régularisation L2 : réduit l'amplitude des coefficients, utile lorsque les variables sont corrélées.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [6]:
m2, p2 = eval_reg(make_pipeline(StandardScaler(), Ridge(alpha=1.0)), 'Ridge')

NameError: name 'eval_reg' is not defined

## 3. Lasso

Régularisation L1 : peut ramener certains coefficients à zéro et effectuer une sélection de variables.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m3, p3 = eval_reg(make_pipeline(StandardScaler(), Lasso(alpha=0.0005, max_iter=20000)), 'Lasso')

## 4. Régression polynomiale

Ajoute des interactions et courbures. La régularisation limite la complexité.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m4, p4 = eval_reg(make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Ridge(alpha=10)), 'Polynomiale + Ridge')

## 5. Arbre de régression

Découpe l'espace des variables en régions ; la profondeur contrôle la complexité.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m5, p5 = eval_reg(DecisionTreeRegressor(max_depth=4, min_samples_leaf=10, random_state=RANDOM_STATE), 'Arbre régression')

## 6. Forêt aléatoire — régression

Moyenne plusieurs arbres entraînés sur des échantillons et sous-ensembles de variables.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [7]:
m6, p6 = eval_reg(RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1), 'Forêt aléatoire')

NameError: name 'eval_reg' is not defined

## 7. Gradient Boosting — régression

Construit des arbres successifs pour corriger les erreurs résiduelles.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m7, p7 = eval_reg(HistGradientBoostingRegressor(max_iter=150, max_leaf_nodes=10, random_state=RANDOM_STATE), 'Gradient Boosting')

## 8. Régression logistique

Classification probabiliste de la direction du rendement (hausse ou non).

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m8, c8 = eval_cls(make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000)), 'Logistique')

## 9. KNN

Classe une journée selon les voisins les plus proches dans l'espace des indicateurs normalisés.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m9, c9 = eval_cls(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9, weights='distance')), 'KNN')

## 10. Naive Bayes

Classificateur probabiliste supposant l'indépendance conditionnelle des variables.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m10, c10 = eval_cls(make_pipeline(StandardScaler(), GaussianNB()), 'Naive Bayes')

## 11. Arbre de décision

Apprend des règles de décision interprétables pour la direction du rendement.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m11, c11 = eval_cls(DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=RANDOM_STATE), 'Arbre classification')

## 12. Forêt aléatoire — classification

Agrège les votes de plusieurs arbres pour stabiliser la classification.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m12, c12 = eval_cls(RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1), 'Forêt classification')

## 13. SVM

Cherche une frontière à marge maximale ; la normalisation est importante.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m13, c13 = eval_cls(make_pipeline(StandardScaler(), SVC(C=1, kernel='rbf')), 'SVM RBF')

## 14. Gradient Boosting — classification

Ajoute séquentiellement des arbres faibles pour améliorer la classification.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [8]:
m14, c14 = eval_cls(HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=10, random_state=RANDOM_STATE), 'Gradient Boosting classification')

NameError: name 'eval_cls' is not defined

## 15. AdaBoost

Repondère progressivement les observations difficiles à classer.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
m15, c15 = eval_cls(AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE), 'AdaBoost')

## 16. XGBoost

Boosting régularisé. La cellule utilise XGBoost si le paquet est installé.

**Cible :** rendement du prochain jour (régression) ou direction du prochain jour (classification), selon l'algorithme.

In [ ]:
try:
    from xgboost import XGBClassifier
    m16, c16 = eval_cls(XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=RANDOM_STATE), 'XGBoost')
except ImportError:
    print('XGBoost non installé : exécuter %pip install xgboost puis relancer cette cellule.')

## 5. Méthodes non supervisées
Pour ces méthodes, les observations sont les indicateurs techniques. La cible future n'est pas utilisée. Les variables sont standardisées.

In [9]:
scaler_unsup = StandardScaler()
X_unsup = scaler_unsup.fit_transform(X)

NameError: name 'X' is not defined

## 17. K-means

Partitionne les journées en trois groupes de profils d'indicateurs.

In [ ]:
km = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE).fit(X_unsup)
print('Effectifs des groupes :', pd.Series(km.labels_).value_counts().sort_index().to_dict())
print('Inertie :', round(km.inertia_, 3))
print('Silhouette :', round(silhouette_score(X_unsup, km.labels_), 4))

## 18. Classification hiérarchique

Fusionne les observations selon leur proximité ; le nombre de groupes est fixé ici à trois.

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage='ward').fit(X_unsup)
print('Effectifs :', pd.Series(agg.labels_).value_counts().sort_index().to_dict())
print('Silhouette :', round(silhouette_score(X_unsup, agg.labels_), 4))

## 19. DBSCAN

Détecte les zones denses et marque certaines journées comme bruit (-1). Les paramètres dépendent de l'échelle des variables.

In [ ]:
db = DBSCAN(eps=1.5, min_samples=10).fit(X_unsup)
labels_db = db.labels_
print('Groupes (bruit inclus) :', pd.Series(labels_db).value_counts().sort_index().to_dict())
valid = labels_db >= 0
if valid.sum() > 1 and len(set(labels_db[valid])) > 1: print('Silhouette sans bruit :', round(silhouette_score(X_unsup[valid], labels_db[valid]),4))

## 20. Mélanges gaussiens

Attribue des probabilités d'appartenance à trois composantes gaussiennes.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=RANDOM_STATE).fit(X_unsup)
labels_gmm = gmm.predict(X_unsup)
print('Effectifs :', pd.Series(labels_gmm).value_counts().sort_index().to_dict())
print('Log-vraisemblance moyenne :', round(gmm.score(X_unsup),4))

## 21. ACP

Projette les indicateurs sur des axes orthogonaux et quantifie la variance conservée.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_unsup)
print('Variance expliquée par composante :', pca.explained_variance_ratio_.round(4))
print('Variance cumulée :', round(pca.explained_variance_ratio_.sum(),4))

## 22. t-SNE

Visualisation non linéaire des voisinages locaux ; les distances globales entre amas sont à interpréter avec prudence.

In [ ]:
n_tsne = min(1000, len(X_unsup))
idx_tsne = np.linspace(0, len(X_unsup)-1, n_tsne, dtype=int)
X_tsne = TSNE(n_components=2, perplexity=min(30, max(5,(n_tsne-1)//3)), init='pca', learning_rate='auto', random_state=RANDOM_STATE).fit_transform(X_unsup[idx_tsne])
plt.figure(figsize=(8,5)); plt.scatter(X_tsne[:,0], X_tsne[:,1], c=km.labels_[idx_tsne], s=12, cmap='viridis'); plt.title('t-SNE des indicateurs NVIDIA (couleur = K-means)'); plt.show()

## 23. UMAP

Réduction non linéaire préservant une structure de voisinage. Installation facultative.

In [ ]:
try:
    import umap
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RANDOM_STATE).fit_transform(X_unsup)
    plt.figure(figsize=(8,5)); plt.scatter(emb[:,0], emb[:,1], c=km.labels_, s=12, cmap='viridis'); plt.title('UMAP — profils NVIDIA'); plt.show()
except ImportError:
    print('UMAP non installé : %pip install umap-learn')

## 24. Isolation Forest

Détecte les journées atypiques à partir des indicateurs, sans supposer qu'elles sont des erreurs.

In [10]:
iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE).fit(X_unsup)
anomalies = iso.predict(X_unsup) == -1
print('Journées atypiques :', int(anomalies.sum()), '/', len(anomalies))
print('Dates atypiques :', list(X.index[anomalies][:15]))

NameError: name 'X_unsup' is not defined

## 6. Bandits et apprentissage par renforcement (illustrations)
Les sections suivantes sont des simulations pédagogiques sur des récompenses dérivées de rendements historiques. Elles ne constituent pas une stratégie de trading exploitable : elles ignorent notamment les coûts de transaction, le slippage, les contraintes de portefeuille et les biais de marché.

## 25. Bandits manchots

Compare trois actions simplifiées : rester en liquidités, exposition longue, exposition inverse. Les récompenses sont simulées à partir du rendement historique d'entraînement.

In [ ]:
rets = yr_train.to_numpy()
probas = [0.50, np.mean(rets > 0), np.mean(rets < 0)]
print('Probabilités empiriques indicatives :', np.round(probas,3))

## 26. Epsilon-greedy

Explore parfois une action aléatoire et exploite sinon l'action au gain moyen estimé le plus élevé.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
rewards = np.array([0.0, *yr_train.to_numpy()])
# Simulation de trois bras Bernoulli à partir de taux indicatifs
arm_p = np.clip([0.50, np.mean(yr_train>0), np.mean(yr_train<0)], 0.01, 0.99)
counts=np.zeros(3); sums=np.zeros(3); eps=0.1
for t in range(1000):
    a = rng.integers(3) if rng.random()<eps else np.argmax((sums+1)/(counts+2))
    r = int(rng.random()<arm_p[a]); counts[a]+=1; sums[a]+=r
print('Choix :', counts.astype(int), '| gains moyens :', np.round(sums/np.maximum(counts,1),3))

## 27. Q-learning

Démonstration tabulaire avec trois états de momentum et trois actions abstraites. L'environnement est simulé, pas un véritable marché.

In [11]:
Q=np.zeros((3,3)); alpha=0.15; gamma=0.95; rng=np.random.default_rng(RANDOM_STATE)
for ep in range(1500):
    s=int(rng.integers(3))
    for _ in range(20):
        a=int(rng.integers(3)) if rng.random()<max(0.05,1-ep/1200) else int(np.argmax(Q[s]))
        ns=int(np.clip(s + rng.choice([-1,0,1]),0,2))
        r=float((a-1)*(ns-1))
        Q[s,a]+=alpha*(r+gamma*np.max(Q[ns])-Q[s,a]); s=ns
print('Table Q simulée :\
', np.round(Q,3))

Table Q simulée : [[8.648 7.668 7.144]
 [7.336 7.939 7.326]
 [7.161 7.612 8.603]]


## 28. SARSA

Comme Q-learning, mais la cible utilise l'action suivante effectivement choisie par la politique exploratoire.

In [12]:
Q_sarsa=np.zeros((3,3)); alpha=0.15; gamma=0.95; rng=np.random.default_rng(RANDOM_STATE)
def act(s,eps): return int(rng.integers(3)) if rng.random()<eps else int(np.argmax(Q_sarsa[s]))
for ep in range(1500):
    eps=max(0.05,1-ep/1200); s=int(rng.integers(3)); a=act(s,eps)
    for _ in range(20):
        ns=int(np.clip(s+rng.choice([-1,0,1]),0,2)); na=act(ns,eps); r=float((a-1)*(ns-1))
        Q_sarsa[s,a]+=alpha*(r+gamma*Q_sarsa[ns,na]-Q_sarsa[s,a]); s,a=ns,na
print('Table SARSA simulée :\
', np.round(Q_sarsa,3))

Table SARSA simulée : [[9.144 7.517 6.864]
 [7.384 7.715 7.125]
 [7.191 7.552 7.775]]


## 29. DQN — cible conceptuelle

Illustration numérique de la cible de Bellman ; aucun réseau neuronal n'est entraîné dans cette cellule.

In [13]:
recompenses=np.array([0.01,-0.02,0.03]); termine=np.array([False,False,True]); q_max_suivant=np.array([0.04,0.02,0.01]); gamma=0.95
cibles=recompenses+gamma*(~termine)*q_max_suivant
print('Cibles DQN conceptuelles :', np.round(cibles,5))

Cibles DQN conceptuelles : [ 0.048 -0.001  0.03 ]


## 7. Comparaison synthétique des modèles supervisés
Les métriques affichées dans les cellules précédentes sont les résultats à reporter. Les modèles doivent être comparés sur la même période test. Un rendement journalier est bruité : un R² faible ou négatif n'est pas surprenant et ne signifie pas qu'un modèle est inutile dans tous les contextes.

In [14]:
# Exemple de visualisation des prédictions de régression si les modèles ont été exécutés.
plt.figure(figsize=(12,5))
plt.plot(yr_test.index, yr_test.values, label="Rendement réel", alpha=.7)
plt.plot(yr_test.index, p6, label="Forêt aléatoire", alpha=.7)
plt.title("Rendement du prochain jour : réel vs prédiction")
plt.ylabel("Rendement")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'yr_test' is not defined

<Figure size 1200x500 with 0 Axes>

## 8. Analyse des limites et conclusion
- **Temporalité :** le découpage respecte le temps, mais une validation walk-forward serait préférable pour mesurer la stabilité.
- **Cible :** le rendement du prochain jour est difficile à prévoir et peut être dominé par le bruit.
- **Fuite de données :** les indicateurs sont calculés à partir du passé et la cible est décalée d'un jour. Ne pas mélanger aléatoirement les lignes.
- **Coûts :** aucune simulation ne prend en compte frais, spread, slippage, fiscalité ou gestion du risque.
- **Interprétation :** une bonne métrique historique ne garantit pas une performance future.
- **Reproductibilité :** les graines aléatoires sont fixées à 42 ; les versions des bibliothèques peuvent modifier légèrement les sorties.

**Conclusion à compléter après exécution :** indiquer les trois modèles les plus performants sur la période test, comparer MAE/RMSE/R² ou Accuracy/F1, puis discuter leur stabilité et leurs limites.